# Sign Language Gesture Segmentation — YOLO26n-seg Training

**Model**: YOLO26n-seg (3M params, 10.2 GFLOPs)  
**Dataset**: 55,500 images, 37 classes (A-Z, 0-9, space)  
**GPU**: P100 recommended (~4-5 min/epoch)  
**Strategy**: Hybrid AdamW → SGD two-phase training  
**Resume**: Automatic checkpoint detection for 12h timeout recovery

### Pipeline
1. **Data Pipeline**: Download → Stratified Split → Mask Generation → YOLO Format
2. **Phase 1**: AdamW (30 epochs) — fast convergence with adaptive LR
3. **Phase 2**: SGD (70 epochs) — fine-tuning for flatter minima & better generalization
4. **Evaluation**: Classification + Detection + Segmentation + Inference Quality Log

### Resume after Kaggle timeout
If the 12-hour session times out before completion:
1. Add the output of this run as a **dataset input** named `yolo26-checkpoint`
2. Re-run the notebook — completed phases are skipped, interrupted phases resume


In [ ]:
import os, sys, shutil
from pathlib import Path

IS_KAGGLE = os.environ.get("KAGGLE_KERNEL_RUN_TYPE") is not None
print(f"Environment: {'Kaggle' if IS_KAGGLE else 'Local'}")

if IS_KAGGLE:
    if not Path("/kaggle/working/project").exists():
        !git clone https://github.com/mfazrinizar/sign-language-gesture-segmentation-yolo26.git /kaggle/working/project
    os.chdir("/kaggle/working/project")
    !pip install -e ./ultralytics -q
    !pip install kagglehub pyyaml scikit-learn tqdm -q
else:
    # Running locally — find project root
    for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
        if (candidate / "src" / "config.py").exists():
            os.chdir(candidate)
            break

print(f"Working directory: {Path.cwd()}")


In [ ]:
# Fix import shadowing: the project's ultralytics/ source directory shadows
# the installed editable package.  Remove project root from sys.path BEFORE
# importing ultralytics, then restore for src.* imports.
project_root = str(Path.cwd())
if project_root in sys.path:
    sys.path.remove(project_root)

import ultralytics
from ultralytics import YOLO
print(f"Ultralytics {ultralytics.__version__}")

# Restore sys.path for src.* imports
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import torch
from src.config import (
    CLASS_FOLDERS, CLASS_NAMES, INDEX_TO_NAME, NUM_CLASSES,
    IMGSZ, PATIENCE, RANDOM_SEED,
    DATA_DIR, RAW_DIR, SPLITS_DIR, YOLO_SEG_DIR,
    CONFIGS_DIR, RESULTS_DIR, MODELS_DIR,
    RAW_COLOR_FOLDER, RAW_BINARY_FOLDER, KAGGLE_DATASET,
)

# Create absolute-path dataset YAML (used by training & evaluation)
import yaml, tempfile
_src_yaml = CONFIGS_DIR / "dataset_seg.yaml"
with open(_src_yaml) as _f:
    _cfg = yaml.safe_load(_f)
_cfg["path"] = str(YOLO_SEG_DIR.resolve())
_tmp = tempfile.NamedTemporaryFile(mode="w", suffix=".yaml", prefix="dataset_seg_", delete=False)
yaml.dump(_cfg, _tmp, default_flow_style=False)
_tmp.close()
DATA_YAML = _tmp.name

print(f"PyTorch {torch.__version__}, CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
print(f"Dataset YAML: {DATA_YAML}")


## 1. Data Pipeline

Download the dataset, create stratified 70/15/15 splits, generate segmentation
masks from binary images via contour extraction, then convert to YOLO format.

> **Kaggle**: Add dataset `ahmedkhanak1995/sign-language-gesture-images-dataset` as input,
> or it will be downloaded via `kagglehub`.


In [ ]:
from src.data.download import download_dataset, setup_raw_data, validate_dataset
from src.data.split import collect_file_list, split_dataset, save_splits, load_splits

# --- Download & Validate ---
kaggle_input = Path("/kaggle/input/sign-language-gesture-images-dataset")
if IS_KAGGLE and kaggle_input.exists():
    downloaded_path = kaggle_input
    print(f"Dataset from Kaggle input: {downloaded_path}")
else:
    downloaded_path = download_dataset()

raw_dir = setup_raw_data(downloaded_path)
stats = validate_dataset(raw_dir)
total = sum(stats['color'].values())
print(f"\nTotal: {total} colour + {sum(stats['binary'].values())} binary images, {NUM_CLASSES} classes")

# --- Stratified Split (70 / 15 / 15) ---
if (SPLITS_DIR / "train.json").exists():
    print("\nSplits already exist, loading...")
    splits = load_splits(SPLITS_DIR)
else:
    filepaths, labels = collect_file_list(raw_dir)
    splits = split_dataset(filepaths, labels)
    save_splits(splits, SPLITS_DIR)

for name, files in splits.items():
    print(f"  {name}: {len(files)} images")


In [ ]:
from src.data.mask_generator import generate_all_masks
from src.data.convert import convert_dataset
from src.data.preprocess import validate_labels

# --- Generate Segmentation Masks (binary → contour polygons → YOLO labels) ---
all_labels_dir = YOLO_SEG_DIR / "all_labels"
if all_labels_dir.exists() and len(list(all_labels_dir.glob("*.txt"))) >= 55000:
    print(f"Masks already generated: {len(list(all_labels_dir.glob('*.txt')))} labels")
else:
    print("Generating segmentation masks from binary images...")
    generate_all_masks(raw_dir)

# --- Convert to YOLO Directory Structure (resize to 224x224) ---
train_imgs = YOLO_SEG_DIR / "images" / "train"
if train_imgs.exists() and len(list(train_imgs.glob("*.jpg"))) >= 38000:
    print("\nDataset already converted:")
    for split in ["train", "val", "test"]:
        n = len(list((YOLO_SEG_DIR / "images" / split).glob("*.jpg")))
        print(f"  {split}: {n} images")
else:
    print("\nConverting to YOLO format...")
    convert_dataset()

# --- Verify label quality ---
print("\nLabel Validation:")
for split in ["train", "val", "test"]:
    labels_dir = YOLO_SEG_DIR / "labels" / split
    lstats = validate_labels(labels_dir)
    print(f"  {split}: {lstats['valid']}/{lstats['total']} valid")

print("\n\u2713 Data pipeline complete!")


## 2. Training — Hybrid AdamW → SGD

| Phase | Optimizer | Epochs | LR | Purpose |
|-------|-----------|--------|-----|----------|
| 1 | AdamW | 30 | 0.001 → cosine | Fast convergence, adaptive per-param LR |
| 2 | SGD | 70 | 0.01 → cosine | Fine-tuning, flatter minima, better generalization |

**Augmentations**: mosaic, mixup, copy-paste, rotation ±15°, HSV shifts.  
**No flips** — horizontal / vertical flips break sign language gesture meaning.

Each phase saves checkpoints every **5 epochs** (`save_period=5`).  
Re-running the notebook skips completed phases and resumes interrupted ones.


In [ ]:
# ══════════════════════════════════════════════════════════════
# Training Configuration
# ══════════════════════════════════════════════════════════════
TOTAL_EPOCHS  = 100
PHASE1_EPOCHS = 30        # AdamW — fast convergence
PHASE2_EPOCHS = 70        # SGD   — fine-tuning
BATCH  = 64 if IS_KAGGLE else 32   # P100 16 GB can handle 64
DEVICE = "0" if torch.cuda.is_available() else "cpu"
WORKERS = 2 if IS_KAGGLE else 8    # Kaggle limits CPU cores

# Separate directories per phase (avoids checkpoint conflicts)
TRAIN_DIR = RESULTS_DIR / "seg"
P1_NAME = "yolo26n-seg-p1"
P2_NAME = "yolo26n-seg-p2"
P1_DIR  = TRAIN_DIR / P1_NAME
P2_DIR  = TRAIN_DIR / P2_NAME
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Shared augmentation kwargs (no flips for sign language)
AUG_KWARGS = dict(
    mosaic=1.0, mixup=0.1, copy_paste=0.1,
    degrees=15.0, translate=0.1, scale=0.3, shear=5.0,
    flipud=0.0, fliplr=0.0,
    hsv_h=0.015, hsv_s=0.5, hsv_v=0.3,
)

# --- Restore checkpoints from a previous Kaggle session ---
if IS_KAGGLE:
    prev_ckpt = Path("/kaggle/input/yolo26-checkpoint")
    if prev_ckpt.exists():
        for src_pt in prev_ckpt.rglob("*.pt"):
            rel = src_pt.relative_to(prev_ckpt)
            dst_pt = TRAIN_DIR / rel
            dst_pt.parent.mkdir(parents=True, exist_ok=True)
            if not dst_pt.exists():
                shutil.copy2(src_pt, dst_pt)
                print(f"Restored: {rel}")
        # Also restore phase markers
        for marker in prev_ckpt.rglob(".phase_complete"):
            rel = marker.relative_to(prev_ckpt)
            dst_m = TRAIN_DIR / rel
            dst_m.parent.mkdir(parents=True, exist_ok=True)
            if not dst_m.exists():
                dst_m.touch()

print(f"Config: {TOTAL_EPOCHS} epochs = {PHASE1_EPOCHS} (AdamW) + {PHASE2_EPOCHS} (SGD)")
print(f"Batch: {BATCH}, ImgSz: {IMGSZ}, Device: {DEVICE}, Workers: {WORKERS}")


In [ ]:
# ══════════════════════════════════════════════════════════════
# Phase 1 — AdamW  (fast convergence)
# ══════════════════════════════════════════════════════════════
P1_MARKER = P1_DIR / ".phase_complete"
P1_LAST   = P1_DIR / "weights" / "last.pt"

if P1_MARKER.exists():
    print("Phase 1 already complete \u2192 skipping")
elif P1_LAST.exists():
    print(f"Resuming Phase 1 from: {P1_LAST}")
    model_p1 = YOLO(str(P1_LAST), task="segment")
    model_p1.train(
        data=DATA_YAML, epochs=PHASE1_EPOCHS, batch=BATCH, imgsz=IMGSZ,
        device=DEVICE, workers=WORKERS, patience=PATIENCE,
        project=str(TRAIN_DIR), name=P1_NAME, exist_ok=True, resume=True,
        optimizer="AdamW", lr0=0.001, lrf=0.01,
        weight_decay=0.0005, warmup_epochs=5, save_period=5,
        plots=True, save=True, val=True, verbose=True,
        **AUG_KWARGS,
    )
    P1_MARKER.touch()
    print("\u2713 Phase 1 complete!")
else:
    print(f"Starting Phase 1 \u2014 AdamW ({PHASE1_EPOCHS} epochs)")
    model_p1 = YOLO("yolo26n-seg.yaml", task="segment")
    model_p1.train(
        data=DATA_YAML, epochs=PHASE1_EPOCHS, batch=BATCH, imgsz=IMGSZ,
        device=DEVICE, workers=WORKERS, patience=PATIENCE,
        project=str(TRAIN_DIR), name=P1_NAME, exist_ok=True, resume=False,
        optimizer="AdamW", lr0=0.001, lrf=0.01,
        weight_decay=0.0005, warmup_epochs=5, save_period=5,
        plots=True, save=True, val=True, verbose=True,
        **AUG_KWARGS,
    )
    P1_MARKER.touch()
    print("\u2713 Phase 1 complete!")


In [ ]:
# ══════════════════════════════════════════════════════════════
# Phase 2 — SGD  (fine-tuning for better generalisation)
# ══════════════════════════════════════════════════════════════
P2_MARKER = P2_DIR / ".phase_complete"
P2_LAST   = P2_DIR / "weights" / "last.pt"

if P2_MARKER.exists():
    print("Phase 2 already complete \u2192 skipping")
elif P2_LAST.exists():
    print(f"Resuming Phase 2 from: {P2_LAST}")
    model_p2 = YOLO(str(P2_LAST), task="segment")
    model_p2.train(
        data=DATA_YAML, epochs=PHASE2_EPOCHS, batch=BATCH, imgsz=IMGSZ,
        device=DEVICE, workers=WORKERS, patience=PATIENCE,
        project=str(TRAIN_DIR), name=P2_NAME, exist_ok=True, resume=True,
        optimizer="SGD", lr0=0.01, lrf=0.01,
        weight_decay=0.0005, warmup_epochs=3, save_period=5,
        plots=True, save=True, val=True, verbose=True,
        **AUG_KWARGS,
    )
    P2_MARKER.touch()
    print("\u2713 Phase 2 complete!")
else:
    # Load best weights from Phase 1
    p1_best = P1_DIR / "weights" / "best.pt"
    p1_last = P1_DIR / "weights" / "last.pt"
    checkpoint = p1_best if p1_best.exists() else p1_last
    if not checkpoint.exists():
        raise FileNotFoundError("No Phase 1 checkpoint found. Run Phase 1 first.")

    print(f"Starting Phase 2 \u2014 SGD ({PHASE2_EPOCHS} epochs)")
    print(f"Loading Phase 1 weights from: {checkpoint}")
    model_p2 = YOLO(str(checkpoint), task="segment")
    model_p2.train(
        data=DATA_YAML, epochs=PHASE2_EPOCHS, batch=BATCH, imgsz=IMGSZ,
        device=DEVICE, workers=WORKERS, patience=PATIENCE,
        project=str(TRAIN_DIR), name=P2_NAME, exist_ok=True, resume=False,
        optimizer="SGD", lr0=0.01, lrf=0.01,
        weight_decay=0.0005, warmup_epochs=3, save_period=5,
        plots=True, save=True, val=True, verbose=True,
        **AUG_KWARGS,
    )
    P2_MARKER.touch()
    print("\u2713 Phase 2 complete!")


In [ ]:
# --- Save Best Model ---
p2_best = P2_DIR / "weights" / "best.pt"
p1_best = P1_DIR / "weights" / "best.pt"
best_src = p2_best if p2_best.exists() else p1_best

if best_src.exists():
    dst = MODELS_DIR / "yolo26n-seg-best.pt"
    shutil.copy2(best_src, dst)
    print(f"Best model saved: {dst}")
    _m = YOLO(str(dst))
    print(f"Parameters: {sum(p.numel() for p in _m.model.parameters()):,}")
else:
    print("WARNING: No best.pt found — check training logs above.")


## 3. Evaluation & Inference Quality Logging

Comprehensive evaluation on the **test set** (8,325 images):

| Category | Metrics |
|----------|---------|
| Classification | Accuracy, Precision, Recall, F1-Score, Specificity |
| Detection | mAP@50, mAP@50:95, Dice, Jaccard/IoU |
| Segmentation | mAP@50, mAP@50:95, Dice, Jaccard/IoU |

Plus per-image inference logging with mask overlays for quality analysis.


In [ ]:
from src.evaluation.metrics import (
    evaluate_model, compute_mask_metrics,
    log_test_predictions, save_evaluation_report,
)

MODEL_PATH = MODELS_DIR / "yolo26n-seg-best.pt"
assert MODEL_PATH.exists(), f"Model not found: {MODEL_PATH}"

# --- YOLO Validation (built-in metrics) ---
print("Running YOLO validation on test set...")
metrics = evaluate_model(MODEL_PATH, data_yaml=DATA_YAML, split="test", device=DEVICE)

# --- Pixel-level Dice & Jaccard ---
print("\nComputing pixel-level Dice & Jaccard (IoU)...")
mask_metrics = compute_mask_metrics(
    MODEL_PATH, split="test", device=DEVICE, max_images=2000,
)
metrics["segmentation"]["dice_coefficient"] = mask_metrics["mean_dice"]
metrics["segmentation"]["jaccard_index_iou"] = mask_metrics["mean_jaccard"]
metrics["segmentation"]["per_class_dice"] = mask_metrics["per_class_dice"]
metrics["segmentation"]["per_class_jaccard"] = mask_metrics["per_class_jaccard"]
metrics["detection"]["dice_coefficient"] = mask_metrics["mean_dice"]
metrics["detection"]["jaccard_index_iou"] = mask_metrics["mean_jaccard"]

# Save JSON report + plots
eval_dir = RESULTS_DIR / "evaluation"
save_evaluation_report(metrics, eval_dir)

# --- Summary ---
print("\n" + "=" * 60)
print("EVALUATION SUMMARY")
print("=" * 60)
if "classification" in metrics:
    c = metrics["classification"]
    print(f"\nClassification:")
    print(f"  Accuracy:    {c['accuracy']:.4f}")
    print(f"  Precision:   {c['precision_macro']:.4f}")
    print(f"  Recall:      {c['recall_macro']:.4f}")
    print(f"  F1-Score:    {c['f1_macro']:.4f}")
    print(f"  Specificity: {c['specificity_macro']:.4f}")
d = metrics["detection"]
print(f"\nDetection:")
print(f"  mAP@50:      {d['mAP50']:.4f}")
print(f"  mAP@50:95:   {d['mAP50_95']:.4f}")
s = metrics["segmentation"]
print(f"\nSegmentation:")
print(f"  mAP@50:      {s['mAP50']:.4f}")
print(f"  mAP@50:95:   {s['mAP50_95']:.4f}")
print(f"  Dice Coeff:  {s.get('dice_coefficient', 'N/A')}")
print(f"  Jaccard/IoU: {s.get('jaccard_index_iou', 'N/A')}")


In [ ]:
# ══════════════════════════════════════════════════════════════
# Per-image inference quality log (test set)
# ══════════════════════════════════════════════════════════════
# Saves:
#   predictions_log.csv      — per-image: GT class, pred class, conf, Dice, IoU
#   predictions_summary.json — per-class accuracy, mean Dice, mean Jaccard
#   overlays/{class}/*.jpg   — original + green pred mask + red GT contour

print("Logging per-image test predictions for quality analysis...")
csv_path = log_test_predictions(
    MODEL_PATH,
    split="test",
    device=DEVICE,
    save_overlays=True,
)
print(f"\nCSV log:  {csv_path}")
print(f"Summary:  {csv_path.parent / 'predictions_summary.json'}")
print(f"Overlays: {csv_path.parent / 'overlays'}")


## 4. Results & Outputs


In [ ]:
import json

# Display evaluation report
report_path = RESULTS_DIR / "evaluation" / "evaluation_report.json"
if report_path.exists():
    with open(report_path) as f:
        report = json.load(f)
    # Show top-level metrics (skip verbose per-class)
    summary = {k: {kk: vv for kk, vv in v.items() if not kk.startswith("per_class")}
               for k, v in report.items()}
    print(json.dumps(summary, indent=2))

# Key output files
print("\n" + "=" * 60)
print("OUTPUT FILES")
print("=" * 60)
outputs = [
    MODELS_DIR / "yolo26n-seg-best.pt",
    RESULTS_DIR / "evaluation" / "evaluation_report.json",
    RESULTS_DIR / "evaluation" / "test_predictions" / "predictions_log.csv",
    RESULTS_DIR / "evaluation" / "test_predictions" / "predictions_summary.json",
]
for p in outputs:
    status = "\u2713" if p.exists() else "\u2717"
    try:
        rel = p.relative_to(Path.cwd())
    except ValueError:
        rel = p
    print(f"  {status} {rel}")

# On Kaggle: copy outputs to /kaggle/working/ for download
if IS_KAGGLE:
    kaggle_out = Path("/kaggle/working")
    for src_file in outputs:
        if src_file.exists():
            shutil.copy2(src_file, kaggle_out / src_file.name)
    # Copy training weights for potential resume
    for phase_dir in [P1_DIR, P2_DIR]:
        weights = phase_dir / "weights"
        marker  = phase_dir / ".phase_complete"
        if weights.exists():
            dst_dir = kaggle_out / phase_dir.name / "weights"
            dst_dir.mkdir(parents=True, exist_ok=True)
            for pt in weights.glob("*.pt"):
                shutil.copy2(pt, dst_dir / pt.name)
        if marker.exists():
            dst_marker = kaggle_out / phase_dir.name / ".phase_complete"
            dst_marker.parent.mkdir(parents=True, exist_ok=True)
            dst_marker.touch()
    print(f"\nOutputs copied to {kaggle_out} for download")

print("\n\u2713 All done!")
